# Project TITAN: Enterprise Retail Intelligence Platform

# Module 5: SQL Business Analysis

## Objective

Perform business analysis using SQL queries to answer real-world retail business questions.

In [2]:
import pandas as pd
import sqlite3
pd.set_option("display.float_format", "{:,.2f}".format)

In [3]:
df = pd.read_csv("../data/cleaned/cleaned_online_retail.csv")
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom


In [10]:
conn = sqlite3.connect("retail.db")

In [16]:
df["Revenue"] = df["Quantity"] * df["Price"]

In [11]:
df.to_sql("retail",conn,if_exists="replace",index=False
)

779495

In [ ]:
# what is a total revenue?
     
     
query = """SELECT
           SUM(Revenue) AS TOTAL_REVENUE FROM retail;"""
           
pd.read_sql(query,conn)

In [19]:
# what is a total numbers of order?

query = """SELECT
           COUNT(DISTINCT Invoice) AS TOTAL_ORDERS 
           FROM retail; """
pd.read_sql(query,conn)

,TOTAL_ORDERS
0,36975


In [ ]:
# total number of cutomers ?
query = """
SELECT
    COUNT(DISTINCT `Customer ID`) AS Total_Customers
FROM retail;
"""

pd.read_sql(query, conn)

,Total_Customers
0,5881


In [ ]:
# top 10 country ?

query = """SELECT Country,SUM(Revenue) AS TOTAL_REVENUE
FROM retail GROUP BY Country ORDER BY TOTAL_REVENUE DESC LIMIT 10;"""

pd.read_sql(query,conn)

In [ ]:
# top 10 product?
query = """SELECT Description,SUM(Revenue) AS TOTAL_REVENUE FROM retail GROUP BY Description ORDER BY TOTAL_REVENUE DESC LIMIT 10;"""

pd.read_sql(query,conn)

,Description,TOTAL_REVENUE
0,REGENCY CAKESTAND 3 TIER,"277,656.25"
1,WHITE HANGING HEART T-LIGHT HOLDER,"247,048.01"
2,"PAPER CRAFT , LITTLE BIRDIE","168,469.60"
3,Manual,"151,777.67"
4,JUMBO BAG RED RETROSPOT,"134,307.44"
5,POSTAGE,"124,648.04"
6,ASSORTED COLOUR BIRD ORNAMENT,"124,351.86"
7,PARTY BUNTING,"103,283.38"
8,MEDIUM CERAMIC TOP STORAGE JAR,"81,416.73"
9,PAPER CHAIN KIT 50'S CHRISTMAS,"76,598.18"


In [27]:
# what is average revenue per order?
query = """
SELECT
    SUM(Revenue) / COUNT(DISTINCT Invoice) AS Average_Order_Value
FROM retail;
"""

pd.read_sql(query, conn)

,Average_Order_Value
0,469.91


In [29]:
# which customer placed more than one order?
query = """
SELECT
    `Customer ID`,
    COUNT(DISTINCT Invoice) AS Orders
FROM retail
GROUP BY `Customer ID`
HAVING COUNT(DISTINCT Invoice) > 1
ORDER BY Orders DESC;
"""

pd.read_sql(query, conn)


,Customer ID,Orders
0,"14,911.00",398
1,"12,748.00",337
2,"17,841.00",211
3,"15,311.00",208
4,"13,089.00",203
...,...,...
4250,"12,375.00",2
4251,"12,365.00",2
4252,"12,363.00",2
4253,"12,355.00",2


In [30]:
# which product s sold the most units ?
query = """
SELECT
    Description,
    SUM(Quantity) AS Total_Quantity
FROM retail
GROUP BY Description
ORDER BY Total_Quantity DESC
LIMIT 10;
"""

pd.read_sql(query, conn)


,Description,Total_Quantity
0,WORLD WAR 2 GLIDERS ASSTD DESIGNS,105185
1,WHITE HANGING HEART T-LIGHT HOLDER,91757
2,"PAPER CRAFT , LITTLE BIRDIE",80995
3,ASSORTED COLOUR BIRD ORNAMENT,78234
4,MEDIUM CERAMIC TOP STORAGE JAR,77916
5,JUMBO BAG RED RETROSPOT,74224
6,BROCADE RING PURSE,70082
7,PACK OF 60 PINK PAISLEY CAKE CASES,54592
8,60 TEATIME FAIRY CAKE CASES,52828
9,PACK OF 72 RETRO SPOT CAKE CASES,45129


In [31]:
query = """
SELECT
    Description,
    AVG(Price) AS Average_Price
FROM retail
GROUP BY Description
ORDER BY Average_Price DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,Description,Average_Price
0,DOTCOM POSTAGE,744.15
1,PICNIC BASKET WICKER 60 PIECES,649.50
2,Adjustment by Peter on Jun 25 2010,243.68
3,VINTAGE BLUE KITCHEN CABINET,214.86
4,Manual,212.60
5,VINTAGE RED KITCHEN CABINET,184.23
6,RUSTIC SEVENTEEN DRAWER SIDEBOARD,158.79
7,GIANT SEVENTEEN DRAWER SIDEBOARD,158.33
8,REGENCY MIRROR WITH SHUTTERS,154.09
9,CHEST NATURAL WOOD 20 DRAWERS,117.50


In [32]:
query = """
SELECT
    Country,
    ROUND(SUM(Revenue),2) AS Revenue
FROM retail
GROUP BY Country
ORDER BY Revenue DESC;
"""

pd.read_sql(query, conn)

,Country,Revenue
0,United Kingdom,"14,389,234.92"
1,EIRE,"616,570.54"
2,Netherlands,"554,038.09"
3,Germany,"425,019.71"
4,France,"348,768.96"
5,Australia,"169,283.46"
6,Spain,"108,332.49"
7,Switzerland,"100,061.94"
8,Sweden,"91,515.82"
9,Denmark,"68,580.69"


## SQL Question 11

### Customer Classification using CASE WHEN

Classify customers into VIP, Premium, and Regular based on total revenue.

In [ ]:
query = """ SELECT `Customer ID`, SUM(Revenue) AS TOTAL_REVENUE, 
CASE
WHEN SUM(Revenue)>= 300000 THEN "LUXURY"  
WHEN SUM(Revenue)>= 50000 THEN "VIP"  
WHEN SUM(Revenue)>= 20000 THEN "PREMIUM"  
ELSE "REGULAR"  
END AS customer_type
FROM retail
GROUP BY `Customer ID`
ORDER BY TOTAL_REVENUE DESC
LIMIT 50;"""

pd.read_sql(query,conn)

## SQL Question 12

### Customers Spending Above Average

In [ ]:
query = """
SELECT
    `Customer ID`,
    SUM(Revenue) AS Total_Revenue
FROM retail

GROUP BY `Customer ID`
HAVING SUM(Revenue) > (SELECT AVG(CustomerRevenue) 
FROM
(SELECTSUM(Revenue) AS CustomerRevenue FROM retailGROUP BY `Customer ID`))
ORDER BY Total_Revenue DESC;
"""

pd.read_sql(query, conn)

,Customer ID,Total_Revenue
0,"18,102.00","580,987.04"
1,"14,646.00","528,602.52"
2,"14,156.00","313,437.62"
3,"14,911.00","291,420.81"
4,"17,450.00","244,784.25"
...,...,...
1155,"15,329.00","2,958.83"
1156,"16,416.00","2,957.88"
1157,"13,012.00","2,957.83"
1158,"13,110.00","2,957.47"


## SQL Question 13

### Customer Revenue Analysis using CTE (WITH Clause)

In [ ]:
query = """
WITH CustomerSales AS
(
SELECT`Customer ID`,SUM(Revenue) AS Total_Revenue
FROM retail
GROUP BY `Customer ID`
)

SELECT *FROM CustomerSales
WHERE Total_Revenue > 20000
ORDER BY Total_Revenue DESC;
"""

pd.read_sql(query, conn)

## Q14. Top 10 Customers by Revenue Rank

### Business Question

Rank customers based on the revenue they generated.

### Business Value

This helps identify VIP customers for loyalty programs, personalized offers, and retention strategies.

In [ ]:
query = """
SELECT `Customer ID`,SUM(Revenue) AS Total_Revenue,
    RANK() OVER(ORDER BY SUM(Revenue) DESC) AS Revenue_Rank
FROM retail
GROUP BY `Customer ID`
LIMIT 10;
"""

top_customers_rank = pd.read_sql(query, conn)

top_customers_rank

In [ ]:
# Question 15
#📝 Business Question

#Who are the Top 3 customers in each country based on revenue?

In [ ]:
query = """
WITH CustomerSales AS
(SELECT Country,`Customer ID`,SUM(Revenue) AS Total_Revenue,
ROW_NUMBER() OVER(PARTITION BY Country ORDER BY SUM(Revenue) DESC) 
AS Rank_No
FROM retail
GROUP BY Country, `Customer ID`)

SELECT *FROM CustomerSales WHERE Rank_No <= 3
ORDER BY Country, Rank_No;
"""

top3_country = pd.read_sql(query, conn)

top3_country

## Q16. Top Selling Product in Each Country

### Business Question

Which product generated the highest revenue in each country?

### Objective

Identify the best-selling product for every country to support inventory planning and regional marketing.

In [47]:
query = """
WITH PRODUCT_REVENUE AS
(
    SELECT
        Country,
        Description,
        SUM(Revenue) AS Total_Revenue,
        ROW_NUMBER() OVER (
            PARTITION BY Country
            ORDER BY SUM(Revenue) DESC
        ) AS Product_Rank
    FROM retail
    GROUP BY Country, Description
)

SELECT
    Country,
    Description,
    Total_Revenue
FROM PRODUCT_REVENUE
WHERE Product_Rank = 1
ORDER BY Total_Revenue DESC;
"""

top_product_country = pd.read_sql(query, conn)

top_product_country

,Country,Description,Total_Revenue
0,United Kingdom,WHITE HANGING HEART T-LIGHT HOLDER,"227,810.16"
1,Germany,POSTAGE,"38,529.20"
2,France,POSTAGE,"24,400.00"
3,EIRE,Manual,"19,558.11"
4,Norway,Manual,"14,756.64"
5,Netherlands,ROUND SNACK BOXES SET OF4 WOODLAND,"13,315.10"
6,Singapore,Manual,"12,158.90"
7,Spain,POSTAGE,"8,927.00"
8,Belgium,POSTAGE,"6,886.00"
9,Switzerland,POSTAGE,"6,661.00"


## Q17. Monthly Revenue Trend

### Business Question

How much revenue did the business generate every month?

### Objective

Analyze monthly sales performance to identify seasonal trends, peak sales months, and low-performing months.

In [51]:
query =  """ SELECT strftime("%Y-%m",InvoiceDate) AS MONTH , 
             ROUND(SUM(Revenue),2) AS TOTAL_REVENUE FROM retail
             GROUP BY strftime("%Y-%m",InvoiceDate)
             ORDER BY MONTH;
        """ 
        
pd.read_sql(query,conn)

,MONTH,TOTAL_REVENUE
0,2009-12,"683,504.01"
1,2010-01,"555,802.67"
2,2010-02,"504,558.96"
3,2010-03,"696,978.47"
4,2010-04,"591,982.00"
5,2010-05,"597,833.38"
6,2010-06,"636,371.13"
7,2010-07,"589,736.17"
8,2010-08,"602,224.60"
9,2010-09,"829,013.95"


## Q18. Monthly Revenue Growth

### Business Question

How did revenue grow or decline compared to the previous month?

### Objective

Measure Month-over-Month (MoM) Revenue Growth to identify business growth trends.

In [23]:
query = """ WITH MONTHLYREVENUE AS (
    SELECT strftime("%Y-%m",InvoiceDate) AS MONTH,ROUND(SUM(Revenue),2) AS Total_Revenue
    FROM retail
    GROUP BY strftime('%Y-%m', InvoiceDate)
    )
    
    SELECT Month,Total_Revenue,
    LAG(Total_Revenue) 
    OVER(
        ORDER BY Month
    ) AS Previous_Month_Revenue

FROM MonthlyRevenue;"""

pd.read_sql(query,conn)

DatabaseError: Execution failed on sql ' WITH MONTHLYREVENUE AS (
    SELECT strftime("%Y-%m",InvoiceDate) AS MONTH,ROUND(SUM(Revenue),2) AS Total_Revenue
    FROM retail
    GROUP BY strftime('%Y-%m', InvoiceDate)
    )

    SELECT Month,Total_Revenue,
    LAG(Total_Revenue) 
    OVER(
        ORDER BY Month
    ) AS Previous_Month_Revenue

FROM MonthlyRevenue;': no such column: Revenue

## Q19. Month-over-Month Revenue Growth (%)

### Business Question

How much did revenue increase or decrease compared to the previous month?

### Objective

Calculate the Month-over-Month (MoM) growth percentage.

In [29]:
query = """
WITH MonthlyRevenue AS
(
    SELECT

        strftime('%Y-%m', InvoiceDate) AS Month,

        ROUND(SUM(Revenue),2) AS Total_Revenue

    FROM retail

    GROUP BY strftime('%Y-%m', InvoiceDate)
)

SELECT

    Month,

    Total_Revenue,

    LAG(Total_Revenue) OVER(
        ORDER BY Month
    ) AS Previous_Month_Revenue,

    ROUND(
        (
            Total_Revenue -
            LAG(Total_Revenue) OVER(ORDER BY Month)
        )
        /
        LAG(Total_Revenue) OVER(ORDER BY Month)
        *100,
    2) AS Growth_Percentage

FROM MonthlyRevenue;
"""

monthly_growth = pd.read_sql(query, conn)

monthly_growth

,Month,Total_Revenue,Previous_Month_Revenue,Growth_Percentage
0,2009-12,"683,504.01",NaN,NaN
1,2010-01,"555,802.67","683,504.01",-18.68
2,2010-02,"504,558.96","555,802.67",-9.22
3,2010-03,"696,978.47","504,558.96",38.14
4,2010-04,"591,982.00","696,978.47",-15.06
5,2010-05,"597,833.38","591,982.00",0.99
6,2010-06,"636,371.13","597,833.38",6.45
7,2010-07,"589,736.17","636,371.13",-7.33
8,2010-08,"602,224.60","589,736.17",2.12
9,2010-09,"829,013.95","602,224.60",37.66


## Q20. Running Total (Cumulative Revenue)

### Business Question

What is the cumulative revenue generated over time?

### Objective

Calculate the running total of revenue month by month to understand business growth over time.

In [28]:
query = """
WITH MonthlyRevenue AS
(
    SELECT

        strftime('%Y-%m', InvoiceDate) AS Month,

        ROUND(SUM(Revenue),2) AS Total_Revenue

    FROM retail

    GROUP BY strftime('%Y-%m', InvoiceDate)
)

SELECT

    Month,

    Total_Revenue,

    SUM(Total_Revenue) OVER
    (
        ORDER BY Month
    ) AS Running_Total

FROM MonthlyRevenue;
"""

running_total = pd.read_sql(query, conn)

running_total

,Month,Total_Revenue,Running_Total
0,2009-12,"683,504.01","683,504.01"
1,2010-01,"555,802.67","1,239,306.68"
2,2010-02,"504,558.96","1,743,865.64"
3,2010-03,"696,978.47","2,440,844.11"
4,2010-04,"591,982.00","3,032,826.11"
5,2010-05,"597,833.38","3,630,659.49"
6,2010-06,"636,371.13","4,267,030.62"
7,2010-07,"589,736.17","4,856,766.79"
8,2010-08,"602,224.60","5,458,991.39"
9,2010-09,"829,013.95","6,288,005.34"


# Q21 Revenue Contribution (%)
# Business Question Which month contributed the highest percentage of the total revenue?

🎯 Business Objective
Which month generated the highest share of revenue?
Which month contributed the least?
Was Christmas season stronger than other months?



In [ ]:
query = """
WITH Monthly_Revenue AS
(
    SELECT

        strftime('%Y-%m', InvoiceDate) AS Month,

        ROUND(SUM(Quantity * Price), 2) AS Total_Revenue

    FROM retail

    GROUP BY strftime('%Y-%m', InvoiceDate)
)

SELECT

    Month,

    Total_Revenue,

    ROUND(
        Total_Revenue * 100.0 /
        SUM(Total_Revenue) OVER(),
        2
    ) AS Revenue_Percentage

FROM Monthly_Revenue

ORDER BY Revenue_Percentage DESC;
"""

revenue_contribution = pd.read_sql(query, conn)

revenue_contribution

In [26]:
df.to_sql(
    "retail",
    conn,
    if_exists="replace",
    index=False
)

779495

## Q22. Customer Lifetime Value (CLV)

### Business Question

Which customers generated the highest total revenue?

### Objective

Calculate Customer Lifetime Value (CLV) to identify high-value customers.

In [30]:
query = """
SELECT

    `Customer ID`,

    ROUND(SUM(Revenue),2) AS Customer_Lifetime_Value,

    COUNT(DISTINCT Invoice) AS Total_Orders,

    ROUND(AVG(Revenue),2) AS Average_Order_Value

FROM retail

GROUP BY `Customer ID`

ORDER BY Customer_Lifetime_Value DESC

LIMIT 10;
"""

customer_clv = pd.read_sql(query, conn)

customer_clv

,Customer ID,Customer_Lifetime_Value,Total_Orders,Average_Order_Value
0,"18,102.00","580,987.04",145,558.64
1,"14,646.00","528,602.52",152,137.16
2,"14,156.00","313,437.62",156,77.62
3,"14,911.00","291,420.81",398,26.30
4,"17,450.00","244,784.25",51,581.44
5,"13,694.00","195,640.69",143,128.80
6,"17,511.00","172,132.87",60,92.15
7,"16,446.00","168,472.50",2,"56,157.50"
8,"16,684.00","147,142.77",55,204.93
9,"12,415.00","144,458.37",28,155.67


## Q23. ABC Customer Analysis

### Business Question

Classify customers into A, B and C categories based on their Lifetime Value.

### Objective

Identify Premium, Medium and Low Value customers for better marketing strategies.

In [31]:
query = """
WITH CustomerRevenue AS
(
    SELECT

        `Customer ID`,

        ROUND(SUM(Revenue),2) AS CLV

    FROM retail

    GROUP BY `Customer ID`
)

SELECT

    `Customer ID`,

    CLV,

    CASE

        WHEN CLV >= 10000 THEN 'A'

        WHEN CLV >= 5000 THEN 'B'

        ELSE 'C'

    END AS Customer_Category

FROM CustomerRevenue

ORDER BY CLV DESC;
"""

abc_customer = pd.read_sql(query, conn)

abc_customer

,Customer ID,CLV,Customer_Category
0,"18,102.00","580,987.04",A
1,"14,646.00","528,602.52",A
2,"14,156.00","313,437.62",A
3,"14,911.00","291,420.81",A
4,"17,450.00","244,784.25",A
...,...,...,...
5876,"16,738.00",3.75,C
5877,"14,095.00",2.95,C
5878,"13,256.00",0.00,C
5879,"14,103.00",0.00,C


## Q24. Executive Dashboard KPIs

### Business Question

Generate all important business KPIs in a single SQL query.

### Objective

Provide management with a one-row executive summary for the dashboard.

In [32]:
query = """
SELECT

    ROUND(SUM(Revenue),2) AS Total_Revenue,

    COUNT(DISTINCT Invoice) AS Total_Orders,

    COUNT(DISTINCT `Customer ID`) AS Total_Customers,

    ROUND(AVG(Revenue),2) AS Average_Order_Value,

    ROUND(SUM(Quantity),2) AS Total_Quantity

FROM retail;
"""

dashboard_kpi = pd.read_sql(query, conn)

dashboard_kpi

,Total_Revenue,Total_Orders,Total_Customers,Average_Order_Value,Total_Quantity
0,"17,374,804.27",36975,5881,22.29,"10,528,705.00"


# SQL Question 25

## Top 5 Customers in Each Country

### Business Question

Who are the Top 5 highest revenue generating customers in every country?

### Objective

Identify the Top 5 customers for each country based on Customer Lifetime Value (CLV).

### Business Value

- Helps regional managers identify their best customers.
- Supports VIP customer programs.
- Improves customer retention strategies.
- Useful for regional sales performance analysis.

In [33]:
query = """
WITH Top_Customer AS
(
    SELECT

        Country,

        `Customer ID`,

        ROUND(SUM(Revenue),2) AS Total_Revenue,

        ROW_NUMBER() OVER
        (
            PARTITION BY Country
            ORDER BY SUM(Revenue) DESC
        ) AS Rank_No

    FROM retail

    GROUP BY
        Country,
        `Customer ID`
)

SELECT

    Country,

    `Customer ID`,

    Total_Revenue,

    Rank_No

FROM Top_Customer

WHERE Rank_No <= 5

ORDER BY
    Country,
    Rank_No;
"""

top5_customers = pd.read_sql(query, conn)

top5_customers

,Country,Customer ID,Total_Revenue,Rank_No
0,Australia,"12,415.00","144,458.37",1
1,Australia,"12,431.00","7,699.79",2
2,Australia,"12,388.00","3,901.11",3
3,Australia,"12,424.00","3,340.03",4
4,Australia,"12,422.00","2,808.10",5
...,...,...,...,...
143,Unspecified,"14,265.00","1,373.35",2
144,Unspecified,"12,363.00",552.00,3
145,Unspecified,"12,743.00",540.13,4
146,Unspecified,"12,351.00",300.93,5
